# Correlation matrix for longitudinal modelling

Longitudinal scores in PCNtoolkit that use **z-gain** or **thrivelines** need an age-to-age **correlation matrix** per region: how strongly normative z-scores correlate between ages.

This notebook shows how to **compute** that matrix with `CorrelationMatrix.compute()` from a longitudinal cohort with predicted z-scores. [Tutorial 14](14_longitudinal_modelling.ipynb) loads a precomputed matrix and uses it for scoring and plots.

After estimation you can reuse the same matrix without keeping the full reference cohort in memory.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pcntoolkit import (
    BLR,
    CorrelationMatrix,
    NormData,
    NormativeModel,
    ZGainScore,
    load_fcon1000,
    load_lnm,
)
from pcntoolkit.math_functions.basis_function import BsplineBasisFunction

DATA_DIR = Path("pcntoolkit_resources") / "data"
COVARIATE = "age"


In [ ]:
def tidy_region_names(df: pd.DataFrame) -> pd.DataFrame:
    metadata = {"sub_id", "age", "sex", "site", "visit", "group"}
    return df.rename(
        columns={
            column: column.replace("&", "_and_").replace("_thickness", "")
            for column in df.columns
            if column not in metadata
        }
    )


load_fcon1000()
load_lnm()

reference_df = tidy_region_names(pd.read_csv(DATA_DIR / "fcon1000.csv"))
lnm_df = tidy_region_names(pd.read_csv(DATA_DIR / "LNM_data.csv"))

# Demo: one region; extend to all regions in your own analysis.
region = "rh_S_front_sup"
if region not in reference_df.columns:
    region = [c for c in reference_df.columns if c not in {"sub_id", "age", "sex", "site"}][0]
response_vars = [region]


## 1. Longitudinal reference cohort with z-scores

`CorrelationMatrix.compute()` needs longitudinal `NormData` (visit labels) with **`Z`** and **`Yhat`** from `model.predict()`. Same fcon1000 → LNM transfer as tutorial 14; here we use **controls** (`control_test`) to estimate the matrix.


In [ ]:
reference_data = NormData.from_dataframe(
    name="reference",
    dataframe=reference_df,
    covariates=["age"],
    batch_effects=["site", "sex"],
    response_vars=response_vars,
    subject_ids="sub_id",
)

fcon1000_model = NormativeModel(
    BLR(
        basis_function_mean=BsplineBasisFunction(degree=3, nknots=5),
        warp_name="warpsinharcsinh",
    ),
    inscaler="standardize",
    outscaler="standardize",
    savemodel=False,
    saveresults=False,
)
fcon1000_model.fit(reference_data)

control_df = lnm_df[lnm_df["group"] == "control"]
control_data = NormData.from_dataframe(
    name="control",
    dataframe=control_df,
    covariates=["age"],
    batch_effects=["site", "sex"],
    response_vars=response_vars,
    subject_ids="sub_id",
    visits="visit",
)
control_adapt, control_test = control_data.train_test_split(splits=[0.2, 0.8])
lnm_model = fcon1000_model.transfer(control_adapt)
lnm_model.predict(control_test)

print("Subjects for matrix estimation:", len(pd.unique(control_test.get_subject_ids())))


## 2. Compute the correlation matrix

`CorrelationMatrix.compute()` pools visit pairs and smooths correlations over age (`bandwidth` in years). The object stores `bandwidth`, `n_subjects`, `estimated_range`, and the `matrix` array.

> **Note:** The public LNM demo has few controls. Sparse cohorts give extrapolated cells — check `estimated_range` and the heatmap.


In [ ]:
corr = CorrelationMatrix.compute(
    control_test,
    bandwidth=3,
    covariate=COVARIATE,
)

print("bandwidth:", corr.bandwidth)
print("n_subjects:", corr.n_subjects)
print("estimated_range:", corr.estimated_range)


## 3. Inspect the matrix

`corr.matrix` is an `xarray` with two age axes. Treat values outside `estimated_range` as extrapolation when you interpret or plot.


In [ ]:
R = corr.matrix.sel(response_vars=region)
age_lo, age_hi = corr.estimated_range
if age_lo is None or age_hi is None:
    age_lo, age_hi = 18, 50
R = R.sel(age_1=slice(age_lo, age_hi), age_2=slice(age_lo, age_hi))

plt.figure(figsize=(5, 4))
plt.imshow(
    R.values[0],
    origin="lower",
    extent=[age_lo, age_hi, age_lo, age_hi],
    vmin=-1,
    vmax=1,
    cmap="coolwarm",
)
plt.colorbar(label="correlation")
plt.xlabel("age (years)")
plt.ylabel("age (years)")
plt.title(f"Age-to-age z correlations — {region}")
plt.show()


## 4. What you can do next

| Goal | API | Where |
|------|-----|--------|
| z-gain scores | `ZGainScore(model, corr).score(data)` | [Tutorial 14](14_longitudinal_modelling.ipynb) |
| Thriveline table | `ZGainScore(...).get_thrivelines()` | 14 |
| Centile + thriveline plot | `plot_thrivelines(model, thrivelines=...)` | 14 |
| Load precomputed pickles | `CorrelationMatrix.load(path)` | 14 |

z-gain and thrivelines follow [Bayer et al., 2026](https://arxiv.org/abs/2601.07591).


In [ ]:
# Minimal downstream check: z-gain on the same control cohort used for estimation.
zgain = ZGainScore(lnm_model, corr)
scores = zgain.score(score_data=control_test)
print(scores.to_dataframe(name="z_gain").head())
